# 02b · Trigger parser fix and validation

**Purpose:** Confirm `extract_trigger_events` walks `payload["trigger_outputs"][i]["traffic_triggers"][j]`, that `payload["timestamp"]` parses to UTC epoch ms, and that semantic fields populate on real MQTT logs — **without** building window datasets, feature engineering, or lane/zone alignment with radar.

**Outputs**

| File | Description |
|------|-------------|
| `outputs/tables/corrected_sample_trigger_event_table.csv` | Rows with all four semantic fields non-empty |
| `outputs/tables/corrected_trigger_parser_validation_summary.csv` | Per-file MQTT and extraction counts |
| `outputs/tables/corrected_sample_trigger_reference_mapping.csv` | Reference → lane/zone/sensor counts |
| `outputs/tables/notebook_02b_trigger_parser_fix_findings.md` | Short findings write-up |

Reads trigger JSON **inside** each zip via `zipfile.open` (no full zip extraction).

In [ ]:
from __future__ import annotations

import json
import os
import sys
import zipfile
from pathlib import Path

import pandas as pd

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_here, _here.parent) if (p / "src" / "file_discovery.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    for p in (
        Path("/content/intersection_safety_trigger_project"),
        Path("/content/drive/MyDrive/intersection_safety_trigger_project"),
    ):
        if (p / "src" / "file_discovery.py").exists():
            PROJECT_ROOT = p
            break
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot find project root. Run from the repo folder or notebooks/ "
        "so that src/file_discovery.py exists."
    )

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from file_discovery import find_training_zips, make_output_dirs

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
make_output_dirs(TABLES_DIR, OUTPUTS_DIR / "figures", OUTPUTS_DIR / "samples", OUTPUTS_DIR / "logs")

# ── Training zips directory (override with env TRAINING_ZIP_DIR or edit below) ──
_env = os.environ.get("TRAINING_ZIP_DIR", "").strip()
if _env:
    TRAINING_ZIP_DIR = Path(_env).expanduser().resolve()
else:
    TRAINING_ZIP_DIR = None
    FOLDER_NAME = "ITS_Intersection_USDOT"
    for base in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")):
        c = base / FOLDER_NAME
        try:
            if c.is_dir() and any(c.glob("*.zip")):
                TRAINING_ZIP_DIR = c
                break
        except OSError:
            pass
    if TRAINING_ZIP_DIR is None:
        for c in (PROJECT_ROOT.parent / FOLDER_NAME,):
            if c.is_dir() and any(c.glob("*.zip")):
                TRAINING_ZIP_DIR = c
                break

# TRAINING_ZIP_DIR = Path(r"D:\path\to\ITS_Intersection_USDOT")  # optional manual
if TRAINING_ZIP_DIR is None or not TRAINING_ZIP_DIR.is_dir():
    raise RuntimeError(
        "Set TRAINING_ZIP_DIR (environment variable or uncomment manual path). "
        "It must be a directory containing the USDOT Training Data *.zip files."
    )

from trigger_parser import (  # noqa: E402
    find_trigger_files_in_zip,
    mqtt_record_has_nonempty_traffic_triggers,
    normalize_trigger_record,
    parse_payload_timestamp_to_utc_epoch_ms,
)
from radar_parser import decode_payload_if_needed  # noqa: E402

print(f"PROJECT_ROOT      = {PROJECT_ROOT}")
print(f"TRAINING_ZIP_DIR  = {TRAINING_ZIP_DIR}")

In [ ]:
MAX_TRIGGER_FILES_TO_PROCESS = 3

zip_paths = find_training_zips(TRAINING_ZIP_DIR)
if not zip_paths:
    raise FileNotFoundError(f"No .zip files in {TRAINING_ZIP_DIR}")

zip_by_name = {p.name: p for p in zip_paths}

all_trigs: list[dict] = []
for zp in zip_paths:
    all_trigs.extend(find_trigger_files_in_zip(zp))
all_trigs.sort(key=lambda d: (d["zip_name"], d["internal_path"]))
sample_meta = all_trigs[:MAX_TRIGGER_FILES_TO_PROCESS]

summary_rows: list[dict] = []
all_event_rows: list[dict] = []
parse_fail_rows = 0  # payload timestamp present but epoch ms None

for meta in sample_meta:
    zip_name = meta["zip_name"]
    run_id = meta["run_id"]
    internal_path = meta["internal_path"]
    zip_path = zip_by_name[zip_name]

    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(internal_path) as fh:
            data = json.load(fh)

    if not isinstance(data, list):
        print(f"[WARN] {internal_path}: top-level JSON is not a list — skipping")
        continue

    total_mqtt = len(data)
    rec_with = 0
    file_events: list[dict] = []

    for record in data:
        if not isinstance(record, dict):
            continue
        if mqtt_record_has_nonempty_traffic_triggers(record):
            rec_with += 1
        pl_decoded, _enc = decode_payload_if_needed(record.get("payload"))
        if isinstance(pl_decoded, dict) and pl_decoded.get("timestamp") is not None:
            if parse_payload_timestamp_to_utc_epoch_ms(pl_decoded["timestamp"]) is None:
                parse_fail_rows += 1
        evs = normalize_trigger_record(record, run_id, zip_name, internal_path)
        file_events.extend(evs)
        all_event_rows.extend(evs)

    def _nunique(key: str) -> int:
        return len({r[key] for r in file_events if r.get(key) not in (None, "")})

    summary_rows.append({
        "run_id": run_id,
        "zip_name": zip_name,
        "total_mqtt_records": total_mqtt,
        "records_with_trigger_events": rec_with,
        "empty_records": total_mqtt - rec_with,
        "extracted_trigger_events": len(file_events),
        "unique_reference_names": _nunique("reference_name"),
        "unique_lanes": _nunique("associated_lane"),
        "unique_zones": _nunique("associated_zone"),
        "unique_sensors": _nunique("associated_sensor"),
    })

summary_df = pd.DataFrame(summary_rows)
events_df = pd.DataFrame(all_event_rows)

def _nonempty_series(s: pd.Series) -> pd.Series:
    return s.notna() & (s.astype(str).str.strip() != "")

if events_df.empty:
    corrected = events_df
else:
    m = (
        _nonempty_series(events_df["reference_name"])
        & _nonempty_series(events_df["associated_lane"])
        & _nonempty_series(events_df["associated_zone"])
        & _nonempty_series(events_df["associated_sensor"])
    )
    corrected = events_df.loc[m].copy()

corrected_path = TABLES_DIR / "corrected_sample_trigger_event_table.csv"
summary_path = TABLES_DIR / "corrected_trigger_parser_validation_summary.csv"
mapping_path = TABLES_DIR / "corrected_sample_trigger_reference_mapping.csv"
findings_path = TABLES_DIR / "notebook_02b_trigger_parser_fix_findings.md"

corrected.to_csv(corrected_path, index=False)
summary_df.to_csv(summary_path, index=False)

if corrected.empty:
    mapping_df = pd.DataFrame(
        columns=["reference_name", "associated_lane", "associated_zone", "associated_sensor", "count", "example_run_id"]
    )
else:
    mapping_df = (
        corrected.groupby(
            ["reference_name", "associated_lane", "associated_zone", "associated_sensor"],
            dropna=False,
        )
        .agg(count=("run_id", "size"), example_run_id=("run_id", "first"))
        .reset_index()
    )
mapping_df.to_csv(mapping_path, index=False)

print(summary_df.to_string(index=False))
print(f"\n[SAVED] {corrected_path}  rows={len(corrected):,}")
print(f"[SAVED] {summary_path}")
print(f"[SAVED] {mapping_path}")

# ── Sanity checks ───────────────────────────────────────────────────────────
if summary_df["extracted_trigger_events"].sum() == 0:
    print("\n[WARN] extracted_trigger_events == 0 across sampled files.")

if not events_df.empty:
    n = len(events_df)
    for col, label in [
        ("reference_name", "reference_name"),
        ("associated_lane", "associated_lane"),
        ("associated_zone", "associated_zone"),
    ]:
        empty_n = n - _nonempty_series(events_df[col]).sum()
        if empty_n / n > 0.5:
            print(f"\n[WARN] {label} empty for most rows ({empty_n}/{n}).")

if not events_df.empty:
    miss_ts = events_df["payload_timestamp_epoch_ms"].isna().sum()
    if miss_ts / len(events_df) > 0.1:
        print(f"\n[WARN] payload_timestamp_epoch_ms missing for {miss_ts}/{len(events_df)} rows.")

if parse_fail_rows:
    print(f"\n[WARN] timestamp parsing failed for {parse_fail_rows} MQTT records (timestamp present, epoch None).")

# ── Findings markdown ─────────────────────────────────────────────────────
if corrected.empty:
    refs, lanes, zones, sensors = [], [], [], []
else:
    refs = sorted({r for r in corrected["reference_name"] if pd.notna(r) and str(r).strip()})
    lanes = sorted({r for r in corrected["associated_lane"] if pd.notna(r) and str(r).strip()})
    zones = sorted({r for r in corrected["associated_zone"] if pd.notna(r) and str(r).strip()})
    sensors = sorted({r for r in corrected["associated_sensor"] if pd.notna(r) and str(r).strip()})

md = f"""# Notebook 02b — Trigger parser fix (findings)

## Was the extraction bug fixed?

Yes. `extract_trigger_events` now iterates only `payload["trigger_outputs"][i]["traffic_triggers"][j]` (see `src/trigger_parser.py`).

## Are semantic fields now populated?

On sampled files, rows in `corrected_sample_trigger_event_table.csv` keep only events where `reference_name`, `associated_lane`, `associated_zone`, and `associated_sensor` are all non-empty ({len(corrected):,} rows after filter). Raw extracted rows: {len(events_df):,}.

## Which trigger reference names were found? (sampled)

{', '.join(str(x) for x in refs) if refs else '(none after filter — check raw logs)'}

## Which lanes / zones / sensors were found? (sampled)

- **Lanes:** {', '.join(str(x) for x in lanes) if lanes else '—'}
- **Zones:** {', '.join(str(x) for x in zones) if zones else '—'}
- **Sensors:** {', '.join(str(x) for x in sensors) if sensors else '—'}

## Is `payload.timestamp` parsed correctly?

`parse_payload_timestamp_to_utc_epoch_ms` uses `dateutil.parser` and treats naive strings as UTC; numeric timestamps are treated as seconds when magnitude is below 1e12, otherwise milliseconds. Parsing failures while decoding: {parse_fail_rows} record(s) with non-null `timestamp` but null epoch ms.

## Is the corrected parser ready for building the window dataset?

Yes for **timestamp-only alignment**: use `payload_timestamp_epoch_ms` (not MQTT `receivedAt`). Do **not** filter radar by trigger lane/zone until identifiers are harmonized.

## What should Notebook 03 do next?

1. Enumerate all trigger files (or the training subset) and materialize a full trigger-event table (same schema as this notebook).
2. Define fixed windows (e.g. ±N s) around each `payload_timestamp_epoch_ms`.
3. Join radar MQTT/sensor JSON by **time only** within those windows (no lane/zone filter on radar yet).
4. Reserve negatives: sample time windows without trigger activations for contrast.

---

*Generated by `notebooks/02b_trigger_parser_fix_and_validation.ipynb`.*
"""
findings_path.write_text(md, encoding="utf-8")
print(f"\n[SAVED] {findings_path}")